In [2]:
from collections import defaultdict
import json
with open("odds_matches.json") as f:
    matches = json.load(f)
bookies = defaultdict(list)
BOOKIES = {
    "163": "eFortuna",
    "165": "STS",
    "502": "LV BET",
    "572": "BETFAN",
    "591": "Superbet",
}
all_bookies = set()
def probablity_home_win(home_odds, away_odds):
    ph_raw = 1.0 / float(home_odds)
    pa_raw = 1.0 / float(away_odds)

    total = ph_raw + pa_raw

    return ph_raw/total
for match in matches:
    home_win = int(match["home-winner"] == "win")
    name = match["name"]
    home_score = int(match["homeResult"])
    away_score = int(match["awayResult"])
    for provider, odds in match["openingsOdds"].items():
        home, away = odds
        if not (float(home) and float(away)):
            continue
        
        all_bookies.add(provider)
        if provider in BOOKIES:
            bookies[provider].append({
                "bookie": BOOKIES[provider],
                "home_win": home_win,
                "home_odds": float(home),
                "away_odds": float(away),
                "BoN": max(home_score, away_score) * 2 - 1,
                "home_prob": probablity_home_win(home, away),
                "name": name
            })

print(len(all_bookies))

5


In [3]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    log_loss,
    brier_score_loss,
    f1_score,
    precision_score,
    recall_score
)
import pandas as pd

for provider, records in bookies.items():
    df = pd.DataFrame(records)
    X = df["home_prob"]
    y = df["home_win"]
    size = len(X)
    auc = roc_auc_score(y, X)
    ll = log_loss(y, X)
    name = BOOKIES[provider]

    print(f"{name}(size={size}): AUC={auc:.3f} Log Loss={ll:.3f}")
    for N in [1, 3, 5]:
        bon = df.loc[df["BoN"] == N]
        size = len(bon)
        X = bon["home_prob"]
        y = bon["home_win"]
        auc = roc_auc_score(y, X)
        ll = log_loss(y, X)
        print(f"Bo{N} (size={size}) AUC={auc:.3f} Log Loss={ll:.3f}")

STS(size=865): AUC=0.792 Log Loss=0.554
Bo1 (size=192) AUC=0.777 Log Loss=0.575
Bo3 (size=520) AUC=0.808 Log Loss=0.539
Bo5 (size=153) AUC=0.761 Log Loss=0.575
LV BET(size=833): AUC=0.788 Log Loss=0.558
Bo1 (size=187) AUC=0.780 Log Loss=0.574
Bo3 (size=496) AUC=0.803 Log Loss=0.545
Bo5 (size=150) AUC=0.752 Log Loss=0.583
eFortuna(size=589): AUC=0.782 Log Loss=0.560
Bo1 (size=168) AUC=0.777 Log Loss=0.576
Bo3 (size=330) AUC=0.798 Log Loss=0.546
Bo5 (size=91) AUC=0.748 Log Loss=0.584
BETFAN(size=861): AUC=0.794 Log Loss=0.551
Bo1 (size=188) AUC=0.777 Log Loss=0.574
Bo3 (size=517) AUC=0.811 Log Loss=0.537
Bo5 (size=156) AUC=0.762 Log Loss=0.571
Superbet(size=849): AUC=0.792 Log Loss=0.554
Bo1 (size=179) AUC=0.779 Log Loss=0.574
Bo3 (size=514) AUC=0.809 Log Loss=0.539
Bo5 (size=156) AUC=0.757 Log Loss=0.581


In [1]:
import pandas as pd


In [3]:
df = pd.read_csv("oddsportal_matches.csv")

In [5]:
df["betfan_margin"] = (1/df["odds1_betfan_close"] + 1/df["odds2_betfan_close"] - 1) * 100

In [9]:
df[df["betfan_margin"] < 0][["odds1_betfan_close", "odds2_betfan_close", "betfan_margin", "url"]]

,odds1_betfan_close,odds2_betfan_close,betfan_margin,url
1909,1.35,3.91,-0.350478,https://www.oddsportal.com/pl/esports/h2h/dplu...
2239,1.83,2.30,-1.876930,https://www.oddsportal.com/pl/esports/h2h/invi...
2515,3.10,3.30,-37.438905,1.3
2519,6.80,7.00,-71.008403,1.06
2522,2.25,2.30,-12.077295,1.6
...,...,...,...,...
9340,3.10,3.00,-34.408602,1.3
9341,2.50,2.40,-18.333333,1.5
9343,5.50,5.00,-61.818182,1.13
9707,3.20,3.20,-37.500000,1.3


In [ ]:
1/1.35 + 1/3.91 - 1